# PathDiff - NIPA 병리 이미지 학습

NIPA 병리 패치 이미지(1024×1024, JPEG)를 이용한 **PathDiff (ControlNet + Class Conditioning)** 학습  
- 데이터: `../../../data/NIPA/origin/` (클래스별 폴더, JPEG + JSON 어노테이션)
- 모델 저장: `../../../model/NIPA/pathdiff/`
- 참고 코드: `PathDiff-main/` (ControlLDM + ControlNet + ControlledUnetModel)
- First Stage: `stabilityai/sd-vae-ft-mse` (diffusers, VAE f8)
- **마스크 조건**: JSON 어노테이션에서 세포 segmentation 마스크 생성 (type_mask 3ch + edge 3ch = 6ch) → ControlNet hint
- **클래스 조건**: 폴더명 기준 클래스 레이블 → ClassEmbedder → cross-attention context (PLIP 텍스트 대체)
- 학습 이미지 크기: 1024×1024

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # GPU 1번 사용

# ── 필수 패키지 설치 (최초 1회만 실행) ────────────────────────────────────────
import subprocess, sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("accelerate")
install_if_missing("diffusers")
install_if_missing("omegaconf")
install_if_missing("einops")
install_if_missing("opencv-python", "cv2")

## 1. 환경 설정 및 Import

In [ ]:
import sys
import json
import math
import random
from pathlib import Path
from glob import glob

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW, lr_scheduler
import torchvision.transforms as T
from torchvision import utils as vutils
from PIL import Image
from tqdm import tqdm
from copy import deepcopy
from accelerate import Accelerator
from diffusers import AutoencoderKL

# PathDiff 모듈 경로 추가
PATHDIFF_ROOT = os.path.dirname(os.path.abspath("__file__"))  # PathDiff-main/
if PATHDIFF_ROOT not in sys.path:
    sys.path.insert(0, PATHDIFF_ROOT)

from omegaconf import OmegaConf
from ldm.modules.diffusionmodules.util import (
    make_beta_schedule, conv_nd, linear, zero_module, timestep_embedding,
)
from ldm.modules.diffusionmodules.openaimodel import (
    UNetModel, TimestepEmbedSequential, ResBlock, Downsample, AttentionBlock,
)
from ldm.modules.attention import SpatialTransformer
from ldm.util import exists

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. 설정값 정의

In [ ]:
# 경로
DATA_ROOT = "../../../data/NIPA/origin"
MODEL_SAVE_DIR = "../../../model/NIPA/pathdiff"
RESULTS_DIR = os.path.join(MODEL_SAVE_DIR, "results")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 이미지 & Latent 크기
IMAGE_SIZE = 1024       # 원본 1024×1024 그대로 사용
LATENT_SIZE = 128       # IMAGE_SIZE // 8 (SD VAE f8)
LATENT_CHANNELS = 4     # SD VAE latent channels
SCALE_FACTOR = 0.18215  # SD VAE 표준 scale factor

# PathDiff 모델 하이퍼파라미터 (mixed_cond config 기준)
MODEL_CHANNELS = 192
CHANNEL_MULT = [1, 2, 3, 5]
NUM_RES_BLOCKS = 2
ATTENTION_RESOLUTIONS = [8, 4, 2]
NUM_HEADS = 1
CONTEXT_DIM = 512       # ClassEmbedder embed_dim (PLIP context_dim 대체)
HINT_CHANNELS = 6       # ControlNet hint: type_mask(3ch) + edge(3ch)

# 학습 설정
BATCH_SIZE = 4
LEARNING_RATE = 3.75e-5 # PathDiff mixed_cond 기본값
TRAIN_NUM_STEPS = 100_000
GRADIENT_ACCUMULATE_EVERY = 1
NUM_WORKERS = 4
TRAIN_RATIO = 0.95

# Noise Schedule (mixed_cond config 기준)
TIMESTEPS = 1000
LINEAR_START = 0.00085
LINEAR_END = 0.0120

# 조건 설정
COND_DROP_PROB = 0.1    # 마스크 + 클래스 unconditional drop
SD_LOCKED = True        # UNet frozen, ControlNet만 학습

# 저장 주기
SAVE_EVERY = 10_000
SAMPLE_EVERY = 5_000
SAVE_LOSS_EVERY = 100

# EMA
EMA_DECAY = 0.9999

# 세포 레이블 정의 (JSON annotation 기준)
CELL_LABEL_MAP = {
    "NT_stroma": 1,
    "NT_epithelial": 2,
    "NT_immune": 3,
    "NT_Muscle": 4,
    "NT_intestinal_metaplasia": 5,
    "TP_invasive": 6,
    "TP_in_situ": 7,
    "Tumor": 8,
    "Tumor_diffuse": 8,
    "Tumor_intestinal": 8,
}

# PathDiff 스타일 색상 맵 (FIXED_COLOR_MAP)
FIXED_COLOR_MAP = {
    0: (0, 0, 0),          # Background - Black
    1: (255, 0, 0),        # NT_stroma - Red
    2: (0, 255, 0),        # NT_epithelial - Green
    3: (0, 0, 255),        # NT_immune - Blue
    4: (255, 255, 0),      # NT_Muscle - Yellow
    5: (255, 0, 255),      # NT_intestinal_metaplasia - Magenta
    6: (0, 255, 255),      # TP_invasive - Cyan
    7: (255, 165, 0),      # TP_in_situ - Orange
    8: (128, 128, 128),    # Tumor - Gray
    9: (255, 255, 255),    # Edge - White
}
NULL_MASK = 10  # PathDiff unconditional mask 값

# 클래스 레이블 정의 (폴더명 기준)
class_dirs = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
CLASS_NAMES = class_dirs
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for unconditional token (index 0)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Classes: {CLASS_NAMES}")
print(f"NUM_CLASSES (incl. uncond): {NUM_CLASSES}")
print(f"Cell labels: {list(CELL_LABEL_MAP.keys())}")
print(f"HINT_CHANNELS: {HINT_CHANNELS} (type_mask 3ch + edge 3ch)")

In [ ]:
CLASS_NAMES

## 3. 데이터셋 정의

- 1024×1024 JPEG를 그대로 사용 (VAE f8 → latent 128×128)
- JSON 어노테이션에서 Polygon 좌표 → **type_mask** (3ch, 색상 코드) + **edge_mask** (3ch, 인스턴스 경계) = **6ch hint**
- PathDiff 스타일: `FIXED_COLOR_MAP`으로 세포 타입별 RGB 색상 할당, edge는 White(255,255,255)
- `COND_DROP_PROB` 확률로 마스크를 NULL_MASK로, 클래스를 0(uncond)으로 드롭 → CFG 지원

In [ ]:
def get_edges(mask_2d):
    """인스턴스 마스크에서 경계(edge) 추출 (PathDiff 동일)"""
    edge = np.zeros_like(mask_2d, dtype=bool)
    edge[:, 1:]  = edge[:, 1:]  | (mask_2d[:, 1:] != mask_2d[:, :-1])
    edge[:, :-1] = edge[:, :-1] | (mask_2d[:, 1:] != mask_2d[:, :-1])
    edge[1:, :]  = edge[1:, :]  | (mask_2d[1:, :] != mask_2d[:-1, :])
    edge[:-1, :] = edge[:-1, :] | (mask_2d[1:, :] != mask_2d[:-1, :])
    return edge.astype(np.float64)


def apply_colormap(mask_2d, color_map):
    """레이블 마스크 → RGB 색상 마스크 (PathDiff FIXED_COLOR_MAP 스타일)"""
    colored = np.zeros((*mask_2d.shape, 3), dtype=np.uint8)
    for label, color in color_map.items():
        colored[mask_2d == label] = color
    return colored


def json_to_masks(json_path, img_size=1024, target_size=1024):
    """
    NIPA JSON 어노테이션 → type_mask(3ch) + edge_mask(3ch) = 6ch hint
    PathDiff PanNukeDatasetV2 스타일로 변환
    """
    with open(json_path) as f:
        data = json.load(f)

    objects = data["content"]["file"]["objects"]

    # type_map: 각 픽셀의 세포 타입 레이블
    # inst_map: 각 픽셀의 인스턴스 ID (경계 추출용)
    type_map = np.zeros((img_size, img_size), dtype=np.uint8)
    inst_map = np.zeros((img_size, img_size), dtype=np.int32)

    for inst_id, obj in enumerate(objects, start=1):
        label_nm = obj["label_nm"]
        coords = np.array(obj["coordinate"], dtype=np.int32)
        cell_label = CELL_LABEL_MAP.get(label_nm, 0)

        cv2.fillPoly(type_map, [coords], cell_label)
        cv2.fillPoly(inst_map, [coords], inst_id)

    # 리사이즈
    type_map = cv2.resize(type_map, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    inst_map = cv2.resize(inst_map, (target_size, target_size), interpolation=cv2.INTER_NEAREST)

    # type_mask: RGB 색상 코드 (3ch)
    type_mask = apply_colormap(type_map, FIXED_COLOR_MAP)

    # edge_mask: 인스턴스 경계 → White(9) (3ch)
    edge_map = get_edges(inst_map).astype(np.uint8)
    edge_map[edge_map == 1] = 9  # edge 레이블 (White)
    edge_mask = apply_colormap(edge_map, FIXED_COLOR_MAP)

    # 6ch concat: [type_mask(3) | edge_mask(3)]
    hint = np.concatenate([type_mask, edge_mask], axis=2)  # (H, W, 6)
    return hint


class NIPAPathDiffDataset(Dataset):
    """NIPA 병리 데이터셋 (PathDiff 스타일: 이미지 + 마스크 hint + 클래스 레이블)"""

    def __init__(self, file_list, labels, image_size=1024, cond_drop_prob=0.1, is_train=True):
        self.file_list = file_list
        self.labels = labels
        self.image_size = image_size
        self.cond_drop_prob = cond_drop_prob
        self.is_train = is_train

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        json_path = os.path.splitext(img_path)[0] + ".json"

        # 이미지 로드 & 전처리 (PathDiff 스타일: /127.5 - 1.0)
        tile = np.array(Image.open(img_path).convert("RGB"))
        tile = cv2.resize(tile, (self.image_size, self.image_size))
        image = (tile / 127.5 - 1.0).astype(np.float32)

        # 마스크 생성 (6ch hint)
        if os.path.exists(json_path):
            hint = json_to_masks(json_path, img_size=1024, target_size=self.image_size)
        else:
            hint = np.full((self.image_size, self.image_size, HINT_CHANNELS), NULL_MASK, dtype=np.uint8)

        # Random flip (image + mask 동시)
        if self.is_train:
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                hint = np.flip(hint, axis=0).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                hint = np.flip(hint, axis=1).copy()

        # 클래스 레이블
        label = self.labels[idx]

        # Unconditional drop (CFG)
        if self.is_train and random.random() < self.cond_drop_prob:
            hint = np.full((self.image_size, self.image_size, HINT_CHANNELS), NULL_MASK, dtype=np.uint8)
            label = 0

        # HWC -> CHW, float
        image = torch.from_numpy(image).permute(2, 0, 1)        # [3, H, W]
        hint = torch.from_numpy(hint.copy()).permute(2, 0, 1).float()  # [6, H, W]

        return image, hint, label


def build_datasets(data_root, class_to_idx, image_size=1024, train_ratio=0.95, cond_drop_prob=0.1):
    all_files, all_labels = [], []
    for cls_name, cls_idx in class_to_idx.items():
        cls_dir = os.path.join(data_root, cls_name)
        files = sorted(glob(os.path.join(cls_dir, "*.jpeg")) + glob(os.path.join(cls_dir, "*.jpg"))
                       + glob(os.path.join(cls_dir, "*.png")))
        all_files.extend(files)
        all_labels.extend([cls_idx] * len(files))
        print(f"  {cls_name} (idx={cls_idx}): {len(files)} images")

    indices = list(range(len(all_files)))
    random.seed(42)
    random.shuffle(indices)
    split = int(len(indices) * train_ratio)
    train_idx, val_idx = indices[:split], indices[split:]

    train_files  = [all_files[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]
    val_files    = [all_files[i] for i in val_idx]
    val_labels   = [all_labels[i] for i in val_idx]

    print(f"\nTotal: {len(all_files)} | Train: {len(train_files)} | Val: {len(val_files)}")

    train_ds = NIPAPathDiffDataset(train_files, train_labels, image_size, cond_drop_prob, is_train=True)
    val_ds   = NIPAPathDiffDataset(val_files,   val_labels,   image_size, cond_drop_prob=0.0, is_train=False)
    return train_ds, val_ds

## 4. 데이터셋 생성 및 확인

In [ ]:
train_dataset, val_dataset = build_datasets(
    DATA_ROOT, CLASS_TO_IDX,
    image_size=IMAGE_SIZE,
    train_ratio=TRAIN_RATIO,
    cond_drop_prob=COND_DROP_PROB,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

sample = train_dataset[0]
print(f"Image shape: {sample[0].shape}, Hint shape: {sample[1].shape}, Label: {sample[2]}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img, hint, label = train_dataset[i * (len(train_dataset) // 4)]
    # 이미지 (좌)
    ax = axes[0][i]
    ax.imshow(((img.permute(1, 2, 0).numpy() + 1.0) / 2.0).clip(0, 1))
    cls_name = [k for k, v in CLASS_TO_IDX.items() if v == label]
    cls_name = cls_name[0] if cls_name else "uncond"
    ax.set_title(f"{cls_name} (label={label})")
    ax.axis("off")
    # type_mask (3ch) (우)
    ax2 = axes[1][i]
    type_mask = hint[:3].permute(1, 2, 0).numpy().astype(np.uint8)
    ax2.imshow(type_mask)
    ax2.set_title("Type Mask (hint[:3])")
    ax2.axis("off")
plt.suptitle("Train Dataset: Image (top) + Type Mask (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. 모델 선언

### 구성 (PathDiff ControlLDM 구조)
| 컴포넌트 | 구현 | 역할 |
|---|---|---|
| **VAE** | `stabilityai/sd-vae-ft-mse` (diffusers) | 이미지 ↔ latent 변환 (frozen) |
| **ControlledUnetModel** | `cldm.cldm.ControlledUnetModel` | latent 공간 noise 예측 (frozen, sd_locked) |
| **ControlNet** | `cldm.cldm.ControlNet` | 마스크 hint(6ch) → UNet 제어 (학습 대상) |
| **ClassEmbedder** | 인라인 구현 | 클래스 레이블 → cross-attention context (PLIP 대체) |
| **DDPM Schedule** | `make_beta_schedule` (PathDiff ldm) | noise schedule 관리 |

In [ ]:
# ── DDPM Noise Schedule (PathDiff mixed_cond config 기본값) ───────────
betas = torch.tensor(
    make_beta_schedule("linear", TIMESTEPS, linear_start=LINEAR_START, linear_end=LINEAR_END),
    dtype=torch.float32,
)
alphas            = 1.0 - betas
alphas_cumprod    = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])

sqrt_alphas_cumprod          = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas_cumprod    = torch.sqrt(1.0 / alphas_cumprod)
sqrt_recipm1_alphas_cumprod  = torch.sqrt(1.0 / alphas_cumprod - 1)

posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
posterior_log_variance_clipped = torch.log(posterior_variance.clamp(min=1e-20))
posterior_mean_coef1 = betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)
posterior_mean_coef2 = (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod)

print(f"Noise schedule: linear, T={TIMESTEPS}, beta=[{LINEAR_START}, {LINEAR_END}]")

In [ ]:
# ── ClassEmbedder (PLIP 대체 - 클래스 레이블 → cross-attention context) ────────
class ClassEmbedder(nn.Module):
    """클래스 레이블 → cross-attention context [B, 1, embed_dim]
    PathDiff의 FrozenCLIPEmbedder(PLIP) 역할을 클래스 임베딩으로 대체"""

    def __init__(self, embed_dim, n_classes):
        super().__init__()
        self.embedding = nn.Embedding(n_classes, embed_dim)

    def forward(self, labels):
        return self.embedding(labels.unsqueeze(1))  # [B] -> [B, 1, embed_dim]


# ── VAE (frozen) ───────────────────────────────────────────────
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)
print(f"VAE loaded (frozen): {sum(p.numel() for p in vae.parameters()) / 1e6:.1f}M params")

# ── ControlledUnetModel (PathDiff cldm.cldm.ControlledUnetModel) ───────
# UNet은 frozen (sd_locked=True), ControlNet output을 skip connection으로 주입받음
from cldm.cldm import ControlledUnetModel, ControlNet

unet = ControlledUnetModel(
    image_size=LATENT_SIZE,
    in_channels=LATENT_CHANNELS,
    out_channels=LATENT_CHANNELS,
    model_channels=MODEL_CHANNELS,
    attention_resolutions=ATTENTION_RESOLUTIONS,
    num_res_blocks=NUM_RES_BLOCKS,
    channel_mult=CHANNEL_MULT,
    num_heads=NUM_HEADS,
    use_spatial_transformer=True,
    transformer_depth=1,
    context_dim=CONTEXT_DIM,
    use_checkpoint=True,
    legacy=False,
).to(device)

# sd_locked: UNet 파라미터를 optimizer에 포함하지 않음 (requires_grad는 유지해야 gradient 흐름 보존)
# ControlledUnetModel.forward() 내부의 torch.no_grad()가 encoder 부분을 처리
# output_blocks는 gradient가 흘러야 ControlNet으로 역전파됨
unet.eval()
print(f"UNet (sd_locked={SD_LOCKED}): {sum(p.numel() for p in unet.parameters()) / 1e6:.1f}M params")

# ── ControlNet (PathDiff cldm.cldm.ControlNet) ───────────────────
# 마스크 hint(6ch) → UNet skip connection에 제어 신호 주입
control_net = ControlNet(
    image_size=LATENT_SIZE,
    in_channels=LATENT_CHANNELS,
    hint_channels=HINT_CHANNELS,
    model_channels=MODEL_CHANNELS,
    attention_resolutions=ATTENTION_RESOLUTIONS,
    num_res_blocks=NUM_RES_BLOCKS,
    channel_mult=CHANNEL_MULT,
    num_heads=NUM_HEADS,
    use_spatial_transformer=True,
    transformer_depth=1,
    context_dim=CONTEXT_DIM,
    use_checkpoint=True,
    legacy=False,
).to(device)
print(f"ControlNet (trainable): {sum(p.numel() for p in control_net.parameters()) / 1e6:.1f}M params")

# ── ClassEmbedder ─────────────────────────────────────────────
class_embedder = ClassEmbedder(embed_dim=CONTEXT_DIM, n_classes=NUM_CLASSES).to(device)
print(f"ClassEmbedder: {NUM_CLASSES} classes, embed_dim={CONTEXT_DIM}")

## 6. 학습 설정 (Accelerator, Optimizer, Scheduler, EMA)

In [ ]:
def cycle(dl):
    while True:
        for data in dl:
            yield data


# Accelerator
accelerator = Accelerator(
    split_batches=True,
    mixed_precision="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATE_EVERY,
)

# 학습 대상: ControlNet + ClassEmbedder
# UNet은 optimizer에 포함하지 않아 가중치 업데이트 안 됨 (gradient는 흐름)
train_params = list(control_net.parameters()) + list(class_embedder.parameters())

optimizer = AdamW(train_params, lr=LEARNING_RATE)

# EMA (ControlNet에 대해)
ema_control = deepcopy(control_net).eval()
for p in ema_control.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def update_ema(ema_model, model, decay=EMA_DECAY):
    for ema_p, model_p in zip(ema_model.parameters(), model.parameters()):
        ema_p.data.mul_(decay).add_(model_p.data, alpha=1 - decay)

# Accelerate wrapping
unet, control_net, class_embedder, optimizer, train_loader, val_loader = accelerator.prepare(
    unet, control_net, class_embedder, optimizer, train_loader, val_loader
)
vae = vae.to(accelerator.device)
ema_control = ema_control.to(accelerator.device)

# Schedule tensors
for name in ['sqrt_alphas_cumprod', 'sqrt_one_minus_alphas_cumprod',
             'sqrt_recip_alphas_cumprod', 'sqrt_recipm1_alphas_cumprod',
             'posterior_mean_coef1', 'posterior_mean_coef2',
             'posterior_log_variance_clipped', 'betas',
             'alphas_cumprod', 'alphas_cumprod_prev']:
    globals()[name] = globals()[name].to(accelerator.device)

train_iter = cycle(train_loader)
print(f"Accelerator ready: {accelerator.device}")
print(f"Trainable params: {sum(p.numel() for p in train_params) / 1e6:.1f}M")

## 7. 학습 함수 및 저장 함수 정의

In [ ]:
def get_vae(vae_model):
    return vae_model.module if hasattr(vae_model, "module") else vae_model


def vae_encode(vae_model, imgs):
    """이미지 [-1,1] → latent (scale 적용). PathDiff는 이미지가 이미 [-1,1]"""
    posterior = get_vae(vae_model).encode(imgs)
    z = posterior.latent_dist.sample() * SCALE_FACTOR
    return z


def vae_decode(vae_model, z):
    """latent → 이미지 [0,1]"""
    decoded = get_vae(vae_model).decode(z / SCALE_FACTOR).sample
    return ((decoded + 1.0) / 2.0).clamp(0, 1)


def q_sample(x_start, t, noise=None):
    """전방 확산: x_0 → x_t"""
    if noise is None:
        noise = torch.randn_like(x_start)
    return (
        sqrt_alphas_cumprod[t][:, None, None, None] * x_start
        + sqrt_one_minus_alphas_cumprod[t][:, None, None, None] * noise
    )


def apply_controlnet(control_model, x_noisy, hint, t, context):
    """
    ControlNet forward: hint(6ch mask) → control features
    hint을 latent 해상도(32x32)로 리사이즈 후 ControlNet에 입력
    """
    # hint: [B, 6, 1024, 1024] -> ControlNet input_hint_block이 stride conv로 축소
    # latent가 128x128이므로 hint를 128*4=512로 맞춤
    # 하지만 latent이 32x32이므로 hint도 32*4=128로 맞춰줌
    hint_resized = F.interpolate(hint, size=(LATENT_SIZE * 4, LATENT_SIZE * 4), mode='bilinear', align_corners=False)
    control = control_model(x=x_noisy, hint=hint_resized, timesteps=t, context=context)
    return control


def p_losses(unet_model, control_model, embedder, x_start, hint, labels, t):
    """PathDiff ControlLDM 스타일 손실 계산"""
    noise = torch.randn_like(x_start)
    x_noisy = q_sample(x_start, t, noise)
    context = embedder(labels)  # [B, 1, context_dim]

    # ControlNet: mask hint → control features
    control = apply_controlnet(control_model, x_noisy, hint, t, context)

    # ControlledUnetModel: control features를 skip connection으로 주입
    noise_pred = unet_model(x_noisy, timesteps=t, context=context, control=control)

    return F.mse_loss(noise_pred, noise)


@torch.no_grad()
def p_sample_ddim_controlnet(unet_model, control_model, x, t, t_prev, context, hint, eta=0.0):
    """DDIM 단일 스텝 (ControlNet 적용)"""
    control = apply_controlnet(control_model, x, hint, t, context)
    eps = unet_model(x, timesteps=t, context=context, control=control)

    alpha_t      = alphas_cumprod[t][:, None, None, None]
    alpha_t_prev = alphas_cumprod_prev[t_prev + 1][:, None, None, None]
    sigma = eta * torch.sqrt((1 - alpha_t_prev) / (1 - alpha_t) * (1 - alpha_t / alpha_t_prev))
    pred_x0 = (x - torch.sqrt(1 - alpha_t) * eps) / torch.sqrt(alpha_t)
    dir_xt  = torch.sqrt(1 - alpha_t_prev - sigma**2) * eps
    noise   = sigma * torch.randn_like(x) if eta > 0 else 0.0
    return torch.sqrt(alpha_t_prev) * pred_x0 + dir_xt + noise


@torch.no_grad()
def ddim_sample_cfg_controlnet(
    unet_model, control_model, embedder, vae_model,
    labels, hint, guidance_scale=9.0, ddim_steps=50, eta=0.0,
):
    """CFG + ControlNet DDIM 샘플링 (PathDiff ControlLDM.log_images 스타일)"""
    B = labels.shape[0]
    device_ = labels.device

    uncond_labels = torch.zeros(B, dtype=torch.long, device=device_)
    context_cond   = embedder(labels)
    context_uncond = embedder(uncond_labels)

    x = torch.randn(B, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device_)
    timesteps = torch.linspace(TIMESTEPS - 1, 0, ddim_steps + 1, dtype=torch.long, device=device_)

    for i in range(ddim_steps):
        t      = timesteps[i].expand(B)
        t_prev = timesteps[i + 1].expand(B)

        # ControlNet은 동일 mask hint 사용 (PathDiff: uc_cat = c_cat)
        control_cond   = apply_controlnet(control_model, x, hint, t, context_cond)
        control_uncond = apply_controlnet(control_model, x, hint, t, context_uncond)

        eps_cond   = unet_model(x, timesteps=t, context=context_cond,   control=control_cond)
        eps_uncond = unet_model(x, timesteps=t, context=context_uncond, control=control_uncond)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        alpha_t      = alphas_cumprod[t][:, None, None, None]
        alpha_t_prev = alphas_cumprod_prev[t_prev + 1][:, None, None, None]
        sigma = eta * torch.sqrt((1 - alpha_t_prev) / (1 - alpha_t) * (1 - alpha_t / alpha_t_prev))
        pred_x0 = (x - torch.sqrt(1 - alpha_t) * eps) / torch.sqrt(alpha_t)
        dir_xt  = torch.sqrt(1 - alpha_t_prev - sigma**2) * eps
        noise   = sigma * torch.randn_like(x) if eta > 0 else 0.0
        x = torch.sqrt(alpha_t_prev) * pred_x0 + dir_xt + noise

    return vae_decode(vae_model, x)


def save_checkpoint(milestone, step, running_loss):
    data = {
        "step": step,
        "loss": running_loss,
        "control_net": accelerator.get_state_dict(control_net),
        "unet": accelerator.get_state_dict(unet),
        "embedder": accelerator.get_state_dict(class_embedder),
        "ema_control": accelerator.get_state_dict(ema_control),
        "optimizer": optimizer.state_dict(),
        "config": {
            "image_size": IMAGE_SIZE, "latent_size": LATENT_SIZE,
            "latent_channels": LATENT_CHANNELS, "scale_factor": SCALE_FACTOR,
            "model_channels": MODEL_CHANNELS, "channel_mult": CHANNEL_MULT,
            "num_res_blocks": NUM_RES_BLOCKS, "attention_resolutions": ATTENTION_RESOLUTIONS,
            "num_heads": NUM_HEADS, "context_dim": CONTEXT_DIM,
            "hint_channels": HINT_CHANNELS, "num_classes": NUM_CLASSES,
            "timesteps": TIMESTEPS, "linear_start": LINEAR_START, "linear_end": LINEAR_END,
            "sd_locked": SD_LOCKED,
        },
    }
    path = os.path.join(MODEL_SAVE_DIR, f"model-{milestone}.pt")
    accelerator.save(data, path)
    print(f"Saved checkpoint: {path}")


def load_checkpoint(milestone):
    path = os.path.join(MODEL_SAVE_DIR, f"model-{milestone}.pt")
    data = torch.load(path, map_location=accelerator.device)
    accelerator.unwrap_model(control_net).load_state_dict(data["control_net"])
    accelerator.unwrap_model(unet).load_state_dict(data["unet"])
    accelerator.unwrap_model(class_embedder).load_state_dict(data["embedder"])
    ema_control.load_state_dict(data["ema_control"])
    if "optimizer" in data:
        optimizer.load_state_dict(data["optimizer"])
    print(f"Resumed from {path} (step={data['step']})")
    return data["step"], data.get("loss", [])

## 8. 학습 실행

체크포인트에서 이어서 학습하려면 `RESUME_MILESTONE`을 설정하세요.

In [ ]:
RESUME_MILESTONE = None   # 예: 3  (model-3.pt 에서 재개)

step = 0
running_loss = []

if RESUME_MILESTONE is not None:
    step, running_loss = load_checkpoint(RESUME_MILESTONE)

with tqdm(initial=step, total=TRAIN_NUM_STEPS, disable=not accelerator.is_main_process) as pbar:
    while step < TRAIN_NUM_STEPS:
        total_loss = 0.0
        control_net.train()
        class_embedder.train()
        unet.eval()  # UNet frozen

        for _ in range(GRADIENT_ACCUMULATE_EVERY):
            imgs, hints, labels = next(train_iter)
            imgs   = imgs.to(accelerator.device)    # [B, 3, 1024, 1024] [-1,1]
            hints  = hints.to(accelerator.device)    # [B, 6, 1024, 1024]
            labels = labels.long().to(accelerator.device)

            with torch.no_grad():
                z = vae_encode(vae, imgs)  # [B, 4, 32, 32]

            t = torch.randint(0, TIMESTEPS, (z.shape[0],), device=z.device)

            with accelerator.accumulate(control_net):
                loss = p_losses(unet, control_net, class_embedder, z, hints, labels, t)
                accelerator.backward(loss)
                accelerator.clip_grad_norm_(train_params, 1.0)
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item()

        # EMA update (ControlNet)
        update_ema(ema_control, accelerator.unwrap_model(control_net))

        avg_loss = total_loss / GRADIENT_ACCUMULATE_EVERY
        step += 1
        pbar.set_postfix(loss=f"{avg_loss:.4f}")
        pbar.update(1)

        if step % SAVE_LOSS_EVERY == 0:
            running_loss.append(avg_loss)

        # 샘플 생성
        if step % SAMPLE_EVERY == 0 and accelerator.is_main_process:
            ema_control.eval()
            # val 데이터에서 샘플 hint 가져오기
            sample_imgs, sample_hints, sample_labels = next(iter(val_loader))
            sample_hints  = sample_hints[:4].to(accelerator.device)
            sample_labels = sample_labels[:4].long().to(accelerator.device)

            samples = ddim_sample_cfg_controlnet(
                unet, ema_control, class_embedder, vae,
                sample_labels, sample_hints, guidance_scale=9.0, ddim_steps=50,
            )
            grid = vutils.make_grid(samples, nrow=4, padding=2)
            vutils.save_image(grid, os.path.join(RESULTS_DIR, f"sample-{step // SAMPLE_EVERY}.png"))

            # 마스크 시각화
            mask_grid = vutils.make_grid(sample_hints[:, :3] / 255.0, nrow=4, padding=2)
            vutils.save_image(mask_grid, os.path.join(RESULTS_DIR, f"mask-{step // SAMPLE_EVERY}.png"))

            # 원본
            orig_grid = vutils.make_grid(((sample_imgs[:4] + 1) / 2).clamp(0, 1), nrow=4, padding=2)
            vutils.save_image(orig_grid, os.path.join(RESULTS_DIR, f"original-{step // SAMPLE_EVERY}.png"))

        if step % SAVE_EVERY == 0:
            save_checkpoint(step // SAVE_EVERY, step, running_loss)

print("Training complete!")

## 9. 학습 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(1, 1, figsize=(8, 5))

ax1.plot(running_loss)
ax1.set_title("Training Loss")
ax1.set_xlabel(f"Step (\u00d7{SAVE_LOSS_EVERY})")
ax1.set_ylabel("MSE Loss")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. 생성된 샘플 이미지 확인

In [ ]:
import matplotlib.pyplot as plt

sample_files   = sorted(glob(os.path.join(RESULTS_DIR, "sample-*.png")))
mask_files     = sorted(glob(os.path.join(RESULTS_DIR, "mask-*.png")))
original_files = sorted(glob(os.path.join(RESULTS_DIR, "original-*.png")))

if sample_files and original_files and mask_files:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(Image.open(original_files[-1]))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(Image.open(mask_files[-1]))
    axes[1].set_title("Mask (ControlNet hint)")
    axes[1].axis("off")
    axes[2].imshow(Image.open(sample_files[-1]))
    axes[2].set_title(f"Generated (Milestone {len(sample_files)})")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No sample images found yet.")

## 11. 클래스별 샘플 생성 (Classifier-Free Guidance + ControlNet)

In [ ]:
GUIDANCE_SCALE = 9.0   # PathDiff 기본값 (unconditional_guidance_scale=9.0)
GEN_CLASS = 1          # 생성할 클래스 index (1 ~ NUM_CLASSES-1)
GEN_N = 4              # 생성 이미지 수

# val 데이터에서 마스크 hint 가져오기
ema_control.eval()
_, gen_hints, _ = next(iter(val_loader))
gen_hints = gen_hints[:GEN_N].to(accelerator.device)
gen_labels = torch.full((GEN_N,), GEN_CLASS, dtype=torch.long, device=accelerator.device)

samples = ddim_sample_cfg_controlnet(
    unet, ema_control, class_embedder, vae,
    gen_labels, gen_hints, guidance_scale=GUIDANCE_SCALE, ddim_steps=50,
)

cls_name = CLASS_NAMES[GEN_CLASS - 1] if GEN_CLASS <= len(CLASS_NAMES) else "unknown"
fig, axes = plt.subplots(2, GEN_N, figsize=(4 * GEN_N, 8))
for i in range(GEN_N):
    # 생성 이미지
    axes[0][i].imshow(samples[i].cpu().permute(1, 2, 0).numpy())
    axes[0][i].set_title(f"{cls_name} (w={GUIDANCE_SCALE})")
    axes[0][i].axis("off")
    # 마스크 hint
    axes[1][i].imshow(gen_hints[i, :3].cpu().permute(1, 2, 0).numpy().astype(np.uint8))
    axes[1][i].set_title("Mask hint")
    axes[1][i].axis("off")
plt.suptitle(f"CFG + ControlNet Samples - Class: {cls_name}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"cfg_controlnet_class{GEN_CLASS}.png"), dpi=150, bbox_inches="tight")
plt.show()

## 12. 최종 모델 저장

In [ ]:
final_data = {
    "step"        : step,
    "loss"        : running_loss,
    "control_net" : accelerator.get_state_dict(control_net),
    "unet"        : accelerator.get_state_dict(unet),
    "embedder"    : accelerator.get_state_dict(class_embedder),
    "ema_control" : accelerator.get_state_dict(ema_control),
    "config": {
        "image_size"          : IMAGE_SIZE,
        "latent_size"         : LATENT_SIZE,
        "latent_channels"     : LATENT_CHANNELS,
        "scale_factor"        : SCALE_FACTOR,
        "model_channels"      : MODEL_CHANNELS,
        "channel_mult"        : CHANNEL_MULT,
        "num_res_blocks"      : NUM_RES_BLOCKS,
        "attention_resolutions": ATTENTION_RESOLUTIONS,
        "num_heads"           : NUM_HEADS,
        "context_dim"         : CONTEXT_DIM,
        "hint_channels"       : HINT_CHANNELS,
        "num_classes"         : NUM_CLASSES,
        "timesteps"           : TIMESTEPS,
        "linear_start"        : LINEAR_START,
        "linear_end"          : LINEAR_END,
        "sd_locked"           : SD_LOCKED,
    },
}

final_path = os.path.join(MODEL_SAVE_DIR, "model-final.pt")
accelerator.save(final_data, final_path)
print(f"Final model saved: {final_path}")
print(f"Total steps: {step}")
print(f"Final loss: {running_loss[-1]:.6f}" if running_loss else "No loss recorded")